In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
import os
print(os.listdir("/content/gdrive/MyDrive/BIGDATA/"))
# Xem thử có thư mục 'gradio_base_pipeline' ở đó không

['Dataset cuối kỳ', 'BIGDATA GIỮA KỲ.docx', 'BCTD.docx', 'BaoCao_TienDo_T2_BigData_PySpark.docx', 'Data_Proresscing_FN.ipynb', 'best_classification_pipeline', 'gradio_ready_regression_pipeline', 'Clustering_FN.ipynb', 'gradio_freight_pipeline_model', 'gradio_rfm_kmeans_pipeline', 'gradio_rfm_bkmeans_pipeline', 'gradio_rfm_gmm_pipeline', 'gradio_fpm_pipeline', 'fp_growth_rules_final', 'Recommendation_FN.ipynb', 'Regression_ FN_PL.ipynb', 'gradio_recommendation_als_pipeline', 'Frequent Pattern Mining_FN_PL.ipynb', 'gradio_base_pipeline', 'gradio_best_model', 'Classsification_FN_PL.ipynb', 'WEB_GRADIO_FN.ipynb']


In [3]:
# Gỡ sạch Spark cũ
!pip uninstall -y pyspark

# Cài bản ổn định với Python 3.12
!pip install pyspark==3.4.1 -q

# Java
!apt-get install openjdk-11-jdk-headless -qq -y

Found existing installation: pyspark 4.0.2
Uninstalling pyspark-4.0.2:
  Successfully uninstalled pyspark-4.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 11.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.4.1 which is incompatible.
Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../openjdk-11-jre-headless_11.0.30+7-1ubuntu1~22.04_amd64.deb ...
Unpacking openjdk-11-jre-headless:amd64 (11.0.30+7-1ubuntu1~22.04) ...
Selecting previously unselected package openjdk-11-jdk-headless:amd64.
Preparing to unpack .../

In [4]:
from pyspark.sql import SparkSession

print("⏳ Starting Spark...")

spark = SparkSession.builder \
    .appName("Olist_App") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print("✅ Spark READY!")

⏳ Starting Spark...
✅ Spark READY!


In [5]:
from pyspark.sql import SparkSession

print("⏳ Starting Spark...")

spark = SparkSession.builder \
    .appName("Olist_App") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("✅ Spark READY!")

⏳ Starting Spark...
✅ Spark READY!


In [ ]:
spark.range(5).show()

In [11]:
!pip install findspark

In [12]:
import findspark
findspark.init()

In [6]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. Danh sách tất cả các cột cần thiết (Đã có distance_km)
columns = [
    'seller_id', 'product_category_name', 'product_id', 'order_id', 'customer_id',
    'order_status', 'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix',
    'customer_city', 'customer_state', 'order_item_id', 'shipping_limit_date',
    'price', 'freight_value', 'payment_sequential', 'payment_type',
    'payment_installments', 'payment_value', 'review_id', 'review_score',
    'review_comment_title', 'review_comment_message', 'review_creation_date',
    'review_answer_timestamp', 'product_name_lenght', 'product_description_lenght',
    'product_photos_qty', 'product_weight_g', 'product_length_cm',
    'product_height_cm', 'product_width_cm', 'product_category_name_english',
    'seller_zip_code_prefix', 'seller_city', 'seller_state', 'geolocation_zip_code_prefix',
    'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state',
    'distance_km'
]

# 2. Cấu hình số dòng cho 4 file khác nhau
file_configs = [
    {"name": "test_data_1k.csv", "n_rows": 1000},
    {"name": "test_data_3k.csv", "n_rows": 3000},
    {"name": "test_data_5k.csv", "n_rows": 5000},
    {"name": "test_data_10k.csv", "n_rows": 10000}
]

# 3. Vòng lặp tạo 4 file
for idx, config in enumerate(file_configs):
    n_rows = config["n_rows"]
    file_name = config["name"]

    # Đổi seed cho mỗi file để dữ liệu không bị giống nhau y hệt
    np.random.seed(42 + idx)

    data = {
        'order_id': [f'order_f{idx}_{i}' for i in range(n_rows)],
        # Tạo tệp khách hàng mua lặp lại (mỗi khách mua trung bình 5-10 đơn)
        'customer_unique_id': [f'cust_unique_f{idx}_{i % (n_rows//5)}' for i in range(n_rows)],
        # Random ngày mua trong khoảng 365 ngày
        'order_purchase_timestamp': [(datetime(2018, 1, 1) + timedelta(days=int(np.random.randint(0, 365)))).strftime('%Y-%m-%d %H:%M:%S') for _ in range(n_rows)],
        'order_delivered_customer_date': [(datetime(2018, 1, 10) + timedelta(days=int(np.random.randint(0, 365)))).strftime('%Y-%m-%d %H:%M:%S') for _ in range(n_rows)],
        'order_estimated_delivery_date': [(datetime(2018, 1, 15) + timedelta(days=int(np.random.randint(0, 365)))).strftime('%Y-%m-%d %H:%M:%S') for _ in range(n_rows)],

        'payment_type': np.random.choice(['credit_card', 'boleto', 'voucher', 'debit_card'], n_rows),
        'customer_state': np.random.choice(['SP', 'RJ', 'MG', 'RS', 'PR'], n_rows),
        'seller_state': np.random.choice(['SP', 'PR', 'RJ', 'SC'], n_rows),

        'price': np.random.uniform(10.0, 500.0, n_rows).round(2),
        'freight_value': np.random.uniform(5.0, 50.0, n_rows).round(2),
        'payment_value': np.random.uniform(15.0, 550.0, n_rows).round(2),

        # Điểm review từ 1 đến 5 để chạy Classification
        'review_score': np.random.choice([1, 2, 3, 4, 5], n_rows),
        'review_comment_message': np.random.choice(['Rất tốt', 'Tệ', 'Giao chậm', 'Tuyệt vời', 'Hàng lỗi', 'Bình thường'], n_rows),

        'product_weight_g': np.random.uniform(100, 5000, n_rows).round(0),
        'product_length_cm': np.random.randint(10, 100, n_rows),
        'product_height_cm': np.random.randint(10, 100, n_rows),
        'product_width_cm': np.random.randint(10, 100, n_rows),

        # Đảm bảo cột hình ảnh là dạng số thuần túy (không dính string)
        'product_photos_qty': np.random.randint(1, 6, n_rows).astype(float),

        # Khoảng cách để chạy Regression
        'distance_km': np.random.uniform(5.0, 2000.0, n_rows).round(2)
    }

    # Điền giá trị rỗng cho các cột không dùng đến
    for col in columns:
        if col not in data:
            data[col] = ''

    # Tạo DataFrame và lưu file
    df = pd.DataFrame(data)[columns]
    df.to_csv(file_name, index=False)

    print(f"✅ Đã tạo file: {file_name} ({n_rows} dòng)")

print("-" * 50)
print("🎉 Đã xong! Giờ bạn có 4 file data sạch sẽ, đầy đủ cột để đem đi test app!")

✅ Đã tạo file: test_data_1k.csv (1000 dòng)
✅ Đã tạo file: test_data_3k.csv (3000 dòng)
✅ Đã tạo file: test_data_5k.csv (5000 dòng)
✅ Đã tạo file: test_data_10k.csv (10000 dòng)
--------------------------------------------------
🎉 Đã xong! Giờ bạn có 4 file data sạch sẽ, đầy đủ cột để đem đi test app!


In [7]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [8]:
!apt-get install openjdk-11-jdk -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jre session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm mesa-utils
The following NEW packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jre session-migration x11-utils
0 upgraded, 18 newly installed, 0 to remove and 2 not upgraded.
Need to get 7,256 kB of archives.
After this operation, 18.4 MB of additional disk 

In [ ]:

import os
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
import gradio as gr
import pandas as pd
import plotly.express as px
import time
import datetime
import findspark
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
from pyspark.sql.functions import col, when, lit
from pyspark.ml.classification import LogisticRegressionModel
import pyspark.sql.functions as F
import plotly.graph_objects as go
import numpy as np
# Import các thư viện cần thiết của PySpark
from pyspark.ml import PipelineModel
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.sql.functions import col, concat_ws, round
from pyspark.sql.types import StructType
from pyspark.ml.feature import IndexToString
# =====================================================================
# PHẦN 0: KHỞI TẠO MÔI TRƯỜNG SPARK
# =====================================================================
findspark.init()
print("⏳ Đang khởi tạo Spark Session...")
spark = SparkSession.builder \
    .appName("Olist_Gradio_App") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
print("✅ Spark đã khởi tạo xong!")


# Bắt buộc: Bạn phải chắc chắn rằng biến 'spark' (SparkSession) đã được khởi tạo trước đó rồi nhé!

# =====================================================================
# PHẦN 1.0: TẢI MASTER DATA VÀ MÔ HÌNH (DÙNG CHO DASHBOARD & CLUSTERING)
# =====================================================================
# 1. Tải Master Data bằng PySpark
try:
    print("⏳ Đang tải Master Data cho Dashboard...")
    master_data_path = "/content/gdrive/MyDrive/BIGDATA/Dataset cuối kỳ/Dataset cuối kỳ/cleaned_master_data.parquet"
    df_master = spark.read.parquet(master_data_path)
    # Lưu ý: Lệnh .count() trong PySpark tốn thời gian hơn len() của Pandas,
    # nếu thấy chậm bạn có thể bỏ phần in tổng số dòng đi.
    print(f"✅ Đã tải thành công Master Data! (Tổng số dòng: {df_master.count()})")
except Exception as e:
    print(f"⚠️ Lỗi khi tải Master Data: {e}")
    df_master = spark.createDataFrame([], StructType([])) # Tạo DF rỗng

# 2. Tải Mô hình K-Means
try:
    print("⏳ Đang tải mô hình K-Means Clustering...")
    kmeans_model_path = "/content/gdrive/MyDrive/BIGDATA/gradio_rfm_kmeans_pipeline"
    kmeans_pipeline = PipelineModel.load(kmeans_model_path)
    print("✅ Đã tải thành công mô hình Clustering!")
except Exception as e:
    print(f"⚠️ Lỗi khi tải mô hình Clustering: {e}")
    kmeans_pipeline = None


# =====================================================================
# PHẦN 1.1: TẢI DỮ LIỆU LUẬT KẾT HỢP (FP-GROWTH) BẰNG PYSPARK
# =====================================================================
fp_rules_path = "/content/gdrive/MyDrive/BIGDATA/fp_growth_rules_final"
try:
    print("⏳ Đang tải dữ liệu Luật kết hợp (FP-Growth)...")
    df_rules = spark.read.parquet(fp_rules_path)

    # Chuyển mảng thành chuỗi cách nhau bằng dấu phẩy và làm tròn số
    df_rules = df_rules.withColumn("antecedent", concat_ws(", ", col("antecedent"))) \
                       .withColumn("consequent", concat_ws(", ", col("consequent"))) \
                       .withColumn("confidence", round(col("confidence"), 3)) \
                       .withColumn("lift", round(col("lift"), 3))

    print("✅ Đã tải thành công dữ liệu FP-Growth!")
except Exception as e:
    print(f"⚠️ Lỗi khi tải dữ liệu FP-Growth: {e}")
    df_rules = spark.createDataFrame([], StructType([]))


# =====================================================================
# PHẦN 1.2: TẢI MÔ HÌNH DỰ ĐOÁN ĐIỂM ĐÁNH GIÁ (CLASSIFICATION)
# =====================================================================
try:
    print("⏳ Đang tải mô hình Dự đoán Review Score...")
    base_pipeline = PipelineModel.load("/content/gdrive/MyDrive/BIGDATA/gradio_base_pipeline")
    classifier_model = LogisticRegressionModel.load("/content/gdrive/MyDrive/BIGDATA/gradio_best_model")
    print("✅ Đã tải thành công bộ đôi mô hình Classification!")
except Exception as e:
    print(f"⚠️ Lỗi khi tải mô hình Review Score: {e}")
    base_pipeline = None
    classifier_model = None


# =====================================================================
# PHẦN 1.3: TẢI MÔ HÌNH DỰ ĐOÁN PHÍ VẬN CHUYỂN (REGRESSION)
# =====================================================================
try:
    print("⏳ Đang tải mô hình Dự đoán Phí Vận Chuyển (Regression)...")
    freight_model_path = "/content/gdrive/MyDrive/BIGDATA/gradio_freight_pipeline_model"
    freight_pipeline = PipelineModel.load(freight_model_path)
    print("✅ Đã tải thành công mô hình Regression!")
except Exception as e:
    print(f"⚠️ Lỗi khi tải mô hình Phí Vận Chuyển: {e}")
    freight_pipeline = None


# =====================================================================
# PHẦN 1.5: TẢI MÔ HÌNH KHUYẾN NGHỊ (ALS) VÀ TỪ ĐIỂN SẢN PHẨM
# =====================================================================
try:
    print("⏳ Đang tải mô hình Khuyến nghị (ALS)...")
    als_pipeline_path = "/content/gdrive/MyDrive/BIGDATA/gradio_recommendation_als_pipeline"
    als_pipeline = PipelineModel.load(als_pipeline_path)
    print("✅ Đã tải thành công mô hình ALS!\n")
except Exception as e:
    print(f"⚠️ Lỗi khi tải mô hình ALS: {e}")
    als_pipeline = None

try:
    print("⏳ Đang tải Từ điển Sản phẩm (để hiện tên Danh mục)...")

    # Đọc file CSV bằng PySpark
    products_csv_path = "/content/gdrive/MyDrive/BIGDATA/Dataset cuối kỳ/Dataset cuối kỳ/olist_products_dataset.csv"

    # option("header", "true"): đọc dòng đầu làm tên cột
    df_products_dict = spark.read.option("header", "true") \
                                 .csv(products_csv_path) \
                                 .select("product_id", "product_category_name")

    # Fill giá trị NaN ("Chưa phân loại") bằng phương thức của PySpark
    df_products_dict = df_products_dict.fillna({"product_category_name": "Chưa phân loại"})

    print("✅ Đã tải thành công Từ điển Sản phẩm!")
except Exception as e:
    print(f"⚠️ Lỗi tải Từ điển (Web sẽ chỉ hiện mã ID): {e}")
    df_products_dict = spark.createDataFrame([], StructType([]))

# =====================================================================
# (Các mô hình khác như Regression, Clustering, ALS tạm thời bỏ qua load ở đây)
# =====================================================================


# =====================================================================
# PHẦN 2: CÁC HÀM XỬ LÝ CHÍNH
# =====================================================================

# 🌟 HÀM MỚI: LÀM SẠCH VÀ ĐẸP BIỂU ĐỒ PLOTLY
def beautify_plotly(fig):
    fig.update_layout(
        template="plotly_white",
        plot_bgcolor="rgba(0,0,0,0)",
        paper_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=20, r=20, t=40, b=20),
        font=dict(family="Inter, sans-serif", size=13),
        hovermode="x unified", # Trải nghiệm Hover vuốt ngang cực xịn
        xaxis=dict(showgrid=False, zeroline=False),
        yaxis=dict(showgrid=True, gridcolor="#f0f0f0", zeroline=False)
    )
    return fig

#------------ 2.0  Dashboard ------------------------------------------------
def load_filtered_dashboard(selected_year, selected_cat):
    # 1. KHỞI TẠO SẴN CÁC BIỂU ĐỒ RỖNG
    fig_revenue = go.Figure().update_layout(title="⚠️ Chưa có dữ liệu Doanh thu")
    fig_review = go.Figure().update_layout(title="⚠️ Chưa có dữ liệu Đánh giá")
    fig_cats = go.Figure().update_layout(title="⚠️ Chưa có dữ liệu Danh mục")
    fig_payment = go.Figure().update_layout(title="⚠️ Chưa có dữ liệu Thanh toán")
    fig_cluster = go.Figure().update_layout(title="⚠️ Chưa có dữ liệu Phân cụm")
    kpi_cust, kpi_ord, kpi_rev = "0", "0", "$0 BRL"

    try:
        if df_master is None or df_master.isEmpty():
            fig_revenue.update_layout(title="⚠️ LỖI: Không có dữ liệu trong df_master (Spark)")
            return kpi_cust, kpi_ord, kpi_rev, fig_revenue, fig_review, fig_cats, fig_payment, fig_cluster

        # Tham chiếu DataFrame từ biến Spark
        df = df_master

        # ==========================================
        # 🌟 LOGIC LỌC DỮ LIỆU BẰNG PYSPARK
        # ==========================================
        if selected_year != "Tất cả":
            # Dùng hàm year() của PySpark để lọc
            df = df.filter(F.year(F.col('order_purchase_timestamp')) == int(selected_year))

        if selected_cat != "Tất cả":
            # Tạo điều kiện (condition) trong PySpark
            cond = (F.col('product_category_name') == selected_cat)
            if 'product_category_name_english' in df.columns:
                cond = cond | (F.col('product_category_name_english') == selected_cat)
            df = df.filter(cond)

        if df.isEmpty():
            fig_revenue.update_layout(title=f"⚠️ Không có dữ liệu cho Năm: {selected_year} - Danh mục: {selected_cat}")
            return kpi_cust, kpi_ord, kpi_rev, fig_revenue, fig_review, fig_cats, fig_payment, fig_cluster

        # ==========================================
        # 2. TÍNH TOÁN KPI (Xử lý trực tiếp trên Spark, rất nhanh)
        # ==========================================
        rev_col = 'payment_value' if 'payment_value' in df.columns else ('price' if 'price' in df.columns else None)

        # Tạo danh sách các phép toán aggregation
        agg_exprs = [
            F.countDistinct('customer_unique_id').alias('cust_count') if 'customer_unique_id' in df.columns else F.lit(0).alias('cust_count'),
            F.countDistinct('order_id').alias('ord_count') if 'order_id' in df.columns else F.lit(0).alias('ord_count')
        ]
        if rev_col:
            agg_exprs.append(F.sum(rev_col).alias('total_rev'))
        else:
            agg_exprs.append(F.lit(0).alias('total_rev'))

        # Chạy Action .collect() để lấy số liệu về UI
        kpi_row = df.select(*agg_exprs).collect()[0]
        kpi_cust = f"{kpi_row['cust_count'] or 0:,.0f}"
        kpi_ord = f"{kpi_row['ord_count'] or 0:,.0f}"
        kpi_rev = f"${kpi_row['total_rev'] or 0:,.0f} BRL"

        # ==========================================
        # 3. VẼ CÁC BIỂU ĐỒ THỐNG KÊ (Groupby bằng Spark -> Vẽ bằng Pandas)
        # ==========================================
        if rev_col and 'order_purchase_timestamp' in df.columns:
            # Gom nhóm theo Tháng trên Spark
            df_monthly_spark = df.na.drop(subset=['order_purchase_timestamp', rev_col]) \
                                 .withColumn('Month', F.date_format('order_purchase_timestamp', 'yyyy-MM')) \
                                 .groupBy('Month').agg(
                                     F.sum(rev_col).alias('Revenue'),
                                     F.countDistinct('order_id').alias('Orders')
                                 ).orderBy('Month')
            # Kéo cục dữ liệu nhỏ về Pandas để vẽ
            df_monthly = df_monthly_spark.toPandas()
            if not df_monthly.empty:
                fig_revenue = px.line(df_monthly, x='Month', y='Revenue', title='📈 Xu hướng Doanh thu theo tháng', markers=True)

        if 'review_score' in df.columns:
            # Thay vì kéo toàn bộ cột review về, đếm luôn trên Spark cho nhẹ
            df_review_counts = df.na.drop(subset=['review_score']).groupBy('review_score').count().orderBy('review_score').toPandas()
            if not df_review_counts.empty:
                # Đổi sang px.bar vì dữ liệu đã được gom nhóm (đếm) sẵn
                fig_review = px.bar(df_review_counts, x='review_score', y='count', title='⭐ Phân phối Điểm đánh giá')

        cat_col = 'product_category_name' if 'product_category_name' in df.columns else ('product_category_name_english' if 'product_category_name_english' in df.columns else None)
        if cat_col:
            top_cats = df.na.drop(subset=[cat_col]).groupBy(cat_col).count().orderBy(F.col('count').desc()).limit(10).toPandas()
            if not top_cats.empty:
                fig_cats = px.bar(top_cats, x=cat_col, y='count', title='🏆 Top 10 Danh mục Bán chạy', text_auto=True)
                fig_cats.update_layout(xaxis_tickangle=-45)

        if 'payment_type' in df.columns:
            payment_counts = df.na.drop(subset=['payment_type']).groupBy('payment_type').count().toPandas()
            if not payment_counts.empty:
                fig_payment = px.pie(payment_counts, values='count', names='payment_type', title='💳 Tỷ lệ Thanh toán', hole=0.3)

        # ==========================================
        # 4. VẼ BIỂU ĐỒ CLUSTERING 3D BẰNG SPARK KMeans
        # ==========================================
        if kmeans_pipeline is not None and 'order_purchase_timestamp' in df.columns and rev_col:
            df_rfm = df.na.drop(subset=['order_purchase_timestamp', 'customer_unique_id', rev_col])
            if not df_rfm.isEmpty():
                # Lấy ngày lớn nhất trong tập dữ liệu
                max_date_row = df_rfm.select(F.max('order_purchase_timestamp')).collect()
                if max_date_row and max_date_row[0][0]:
                    max_date = max_date_row[0][0]

                    # Tính RFM hoàn toàn bằng Spark
                    rfm = df_rfm.groupBy('customer_unique_id').agg(
                        F.datediff(F.lit(max_date), F.max('order_purchase_timestamp')).cast('int').alias('Recency'),
                        F.countDistinct('order_id').cast('int').alias('Frequency'),
                        F.sum(rev_col).cast('double').alias('Monetary')
                    ).na.drop()

                    # Tính Log biến đổi
                    rfm = rfm.withColumn('Log_R', F.log(F.col('Recency') + 1)) \
                             .withColumn('Log_F', F.log(F.col('Frequency') + 1)) \
                             .withColumn('Log_M', F.log(F.col('Monetary') + 1))

                    # Dự đoán qua Pipeline
                    preds = kmeans_pipeline.transform(rfm)

                    # Kéo 10,000 dòng kết quả về Pandas để vẽ đồ thị 3D (tránh sập trình duyệt)
                    rfm_result = preds.select("Recency", "Frequency", "Monetary", "Cluster_KMeans").limit(10000).toPandas()

                    if not rfm_result.empty:
                        rfm_result['Phân Loại'] = rfm_result['Cluster_KMeans'].apply(lambda x: f"Cụm Khách Hàng {x}")
                        fig_cluster = px.scatter_3d(
                            rfm_result, x='Recency', y='Frequency', z='Monetary', color='Phân Loại',
                            title="🌌 Phân cụm Khách hàng Toàn Hệ thống (K-Means)", opacity=0.7
                        )
                        fig_cluster.update_layout(margin=dict(l=0, r=0, b=0, t=30))

    except Exception as e:
        print(f"⚠️ LỖI NGHIÊM TRỌNG DASHBOARD: {str(e)}")
        fig_revenue.update_layout(title=f"⚠️ Lỗi xử lý: {str(e)}")

    return kpi_cust, kpi_ord, kpi_rev, fig_revenue, fig_review, fig_cats, fig_payment, fig_cluster

# --- 2.1 HÀM XỬ LÝ DỮ LIỆU FP-GROWTH ---
def real_fp_rules(min_confidence, min_lift): # <--- 1. Thêm tham số min_lift
    if df_rules is None or df_rules.isEmpty():
        return pd.DataFrame({"Lỗi": ["Không tìm thấy dữ liệu!"]})

    # 2. Lọc dữ liệu trên Spark bằng cả 2 điều kiện (Confidence và Lift)
    filtered_df = df_rules.filter(
        (F.col('confidence') >= min_confidence) &
        (F.col('lift') >= min_lift)
    ).orderBy(F.col('lift').desc(), F.col('confidence').desc()).limit(50)
    # ^ 3. Sắp xếp ưu tiên theo Lift giảm dần, sau đó mới đến Confidence

    # Đưa về Pandas hiển thị lên bảng Gradio
    display_df = filtered_df.toPandas()

    # (Tùy chọn) Thêm dòng này để phòng trường hợp lọc gắt quá không ra kết quả nào
    if display_df.empty:
        return pd.DataFrame({"Thông báo": ["Không tìm thấy quy luật nào thỏa mãn điều kiện lọc của bạn!"]})

    display_df = display_df.rename(columns={
        "antecedent": "Sản phẩm đã mua (Antecedent)",
        "consequent": "Sẽ mua kèm (Consequent)",
        "confidence": "Độ tự tin (Confidence)",
        "lift": "Mức độ tăng (Lift)"
    })

    return display_df[["Sản phẩm đã mua (Antecedent)", "Sẽ mua kèm (Consequent)", "Độ tự tin (Confidence)", "Mức độ tăng (Lift)"]]


# --- 2.2 HÀM DỰ ĐOÁN REVIEW SCORE ---
def real_classify_review(comment, purchase_date, delivered_date, estimated_date, payment_type, state, freight, photos, weight):
    if base_pipeline is None or classifier_model is None:
        return {"Lỗi: Chưa tải được mô hình": 1.0}

    try:
        # Bỏ qua bước tạo pd.DataFrame, tạo thẳng Spark DataFrame từ List Dict
        real_data = [{
            "review_comment_message": comment if comment else "",
            "order_purchase_timestamp": purchase_date,
            "order_delivered_customer_date": delivered_date,
            "order_estimated_delivery_date": estimated_date,
            "payment_type": payment_type,
            "customer_state": state,
            "freight_value": float(freight),
            "product_photos_qty": float(photos),
            "product_weight_g": float(weight)
        }]

        df_input = spark.createDataFrame(real_data)
        df_input = df_input.selectExpr(
            "review_comment_message",
            "cast(order_purchase_timestamp as timestamp)",
            "cast(order_delivered_customer_date as timestamp)",
            "cast(order_estimated_delivery_date as timestamp)",
            "payment_type", "customer_state",
            "cast(freight_value as double)",
            "cast(product_photos_qty as double)",
            "cast(product_weight_g as double)"
        )

        ready_data = base_pipeline.transform(df_input)
        prediction_df = classifier_model.transform(ready_data)

        pred_val = prediction_df.select("prediction").collect()[0][0]
        prob_array = prediction_df.select("probability").collect()[0][0]

        if pred_val == 1.0:
            return {"⭐⭐⭐⭐⭐ Đơn hàng TÍCH CỰC (>= 4 sao)": float(prob_array[1])}
        else:
            return {"⚠️ Đơn hàng TIÊU CỰC (=< 3 sao)": float(prob_array[0])}

    except Exception as e:
        return {f"Hệ thống lỗi (Kiểm tra định dạng ngày): {str(e)}": 1.0}


# --- 2.3  HÀM DỰ ĐOÁN PHÍ VẬN CHUYỂN (REGRESSION) ---
def real_predict_freight(weight, distance, price, length, height, width, cust_state, sell_state):
    if freight_pipeline is None:
        return "⚠️ Lỗi: Chưa tải được mô hình Regression!"

    try:
        real_data = [{
            "product_weight_g": float(weight),
            "distance_km": float(distance),
            "price": float(price),
            "product_length_cm": float(length),
            "product_height_cm": float(height),
            "product_width_cm": float(width),
            "customer_state": cust_state,
            "seller_state": sell_state
        }]

        # Tạo thẳng Spark DataFrame
        df_input = spark.createDataFrame(real_data)

        df_input = df_input.selectExpr(
            "cast(product_weight_g as double)",
            "cast(distance_km as double)",
            "cast(price as double)",
            "cast(product_length_cm as double)",
            "cast(product_height_cm as double)",
            "cast(product_width_cm as double)",
            "customer_state",
            "seller_state"
        )

        prediction_df = freight_pipeline.transform(df_input)
        pred_val = prediction_df.select("prediction").collect()[0][0]

        return f"💰 {pred_val:.2f} BRL"

    except Exception as e:
        return f"⚠️ Lỗi xử lý: {str(e)}"


# --- 2.4 CLUSTERING (Phân cụm khách hàng từ File) ---
def real_clustering(file_obj):
    if kmeans_pipeline is None or file_obj is None:
        return None, None, pd.DataFrame({"Thông báo": ["Vui lòng tải file hoặc chờ mô hình load."]})

    try:
        # 1. Đọc file CSV thẳng bằng PySpark (thay vì Pandas)
        df_input = spark.read.option("header", "true") \
                             .option("inferSchema", "true") \
                             .csv(file_obj.name)

        predictions = kmeans_pipeline.transform(df_input)

        # 2. Tạo Summary Bảng Đặc Điểm Nhóm (Tính trên Spark cho nhẹ)
        summary_spark = predictions.groupBy(F.col("Cluster_KMeans").alias("Cụm Khách Hàng")).agg(
            F.count("customer_unique_id").alias("Số Lượng KH"),
            F.mean("Recency").alias("Ngày mua gần nhất (Recency TB)"),
            F.mean("Frequency").alias("Tần suất mua (Frequency TB)"),
            F.mean("Monetary").alias("Chi tiêu (Monetary TB)")
        ).orderBy("Cụm Khách Hàng")

        # Đem kết quả thống kê nhỏ này về Pandas
        summary_df = summary_spark.toPandas()
        summary_df['Cụm Khách Hàng'] = summary_df['Cụm Khách Hàng'].apply(lambda x: f"Nhóm {x}")
        summary_df = summary_df.round(2)

        # 3. Kéo giới hạn 5000 dòng về Pandas để vẽ bảng và biểu đồ (Chống treo web)
        full_result = predictions.select(
            F.col("Cluster_KMeans").alias("Cụm"),
            F.col("customer_unique_id").alias("Mã Khách Hàng"),
            "Recency", "Frequency", "Monetary"
        ).limit(5000).toPandas()

        full_result['Phân Loại'] = full_result['Cụm'].apply(lambda x: f"Nhóm {x}")

        # 4. Vẽ biểu đồ 3D RFM
        fig = px.scatter_3d(
            full_result,
            x='Recency', y='Frequency', z='Monetary',
            color='Phân Loại',
            title="Trực quan hóa không gian khách hàng 3D (R, F, M)",
            opacity=0.7,
            labels={'Recency': 'Ngày mua gần nhất', 'Frequency': 'Tần suất', 'Monetary': 'Số tiền'}
        )
        fig.update_layout(margin=dict(l=0, r=0, b=0, t=30))

        # Trình bày bảng chi tiết
        final_table = full_result[["Mã Khách Hàng", "Recency", "Frequency", "Monetary", "Phân Loại"]]

        return final_table, fig, summary_df

    except Exception as e:
        return pd.DataFrame({"Lỗi": [str(e)]}), None, pd.DataFrame()


#---------------------------------------------------------------------

def real_fp_rules(min_confidence, min_lift, search_term=""):
    """
    Lọc và hiển thị luật kết hợp FP-Growth.
    - Lọc theo Confidence và Lift
    - Tìm kiếm theo tên sản phẩm (antecedent/consequent)
    - Trả về: bảng luật + biểu đồ phân phối Lift + KPI summary
    """
    # Kiểm tra dữ liệu đã tải chưa (df_rules là biến global từ phần load)
    if df_rules is None or df_rules.isEmpty():
        empty_df = pd.DataFrame({"Lỗi": ["Không tìm thấy dữ liệu FP-Growth!"]})
        return empty_df, go.Figure().update_layout(title="Không có dữ liệu"), "0", "0", "0", "0"

    try:
        # --- Bước 1: Lọc trên Spark ---
        filtered = df_rules.filter(
            (F.col('confidence') >= min_confidence) &
            (F.col('lift') >= min_lift)
        )

        # Tìm kiếm theo từ khoá (nếu có)
        if search_term and search_term.strip():
            term = search_term.strip().lower()
            filtered = filtered.filter(
                F.lower(F.col('antecedent')).contains(term) |
                F.lower(F.col('consequent')).contains(term)
            )

        # --- Bước 2: Tính KPI ---
        total_rules = df_rules.count()
        filtered_count = filtered.count()

        if filtered_count == 0:
            return (
                pd.DataFrame({"Thông báo": ["Không có luật nào thỏa điều kiện!"]}),
                go.Figure().update_layout(title="Không có dữ liệu để vẽ biểu đồ"),
                str(total_rules), "0", "N/A", "N/A"
            )

        kpi_row = filtered.select(
            F.mean('lift').alias('avg_lift'),
            F.mean('confidence').alias('avg_conf')
        ).collect()[0]
        avg_lift = f"{kpi_row['avg_lift']:.3f}" if kpi_row['avg_lift'] else "N/A"
        avg_conf = f"{kpi_row['avg_conf']:.3f}" if kpi_row['avg_conf'] else "N/A"

        # --- Bước 3: Lấy top 50 để hiển thị bảng ---
        display_spark = filtered.orderBy(
            F.col('lift').desc(), F.col('confidence').desc()
        ).limit(50)

        display_df = display_spark.toPandas()

        # Thêm badge phân loại mức độ
        def classify_lift(l):
            if l > 4: return "🟢 Rất cao"
            if l > 3: return "🔵 Cao"
            if l > 2: return "🟡 Trung bình"
            return "⚪ Thấp"

        display_df['Mức độ'] = display_df['lift'].apply(classify_lift)
        display_df = display_df.rename(columns={
            "antecedent": "Sản phẩm đã mua",
            "consequent": "Sẽ mua kèm",
            "confidence": "Confidence",
            "lift": "Lift"
        })

        # --- Bước 4: Vẽ biểu đồ phân phối Lift ---
        # Lấy tất cả giá trị lift từ kết quả lọc (giới hạn 2000 dòng để vẽ nhanh)
        lift_vals = filtered.select('lift').limit(2000).toPandas()['lift']

        fig = go.Figure()
        fig.add_trace(go.Histogram(
            x=lift_vals,
            nbinsx=20,
            name='Phân phối Lift',
            marker_color='#378add',
            opacity=0.8
        ))
        fig.add_vline(
            x=lift_vals.mean(),
            line_dash="dash",
            line_color="#ef9f27",
            annotation_text=f"TB: {lift_vals.mean():.2f}",
            annotation_position="top right"
        )
        fig.update_layout(
            title=f"📊 Phân phối Lift ({filtered_count} luật sau lọc)",
            xaxis_title="Lift",
            yaxis_title="Số lượng luật",
            template="plotly_white",
            plot_bgcolor="rgba(0,0,0,0)",
            paper_bgcolor="rgba(0,0,0,0)",
            margin=dict(l=20, r=20, t=50, b=30),
            font=dict(family="Inter, sans-serif", size=13),
            showlegend=False
        )

        final_cols = ["Sản phẩm đã mua", "Sẽ mua kèm", "Confidence", "Lift", "Mức độ"]
        return (
            display_df[final_cols],
            fig,
            str(total_rules),
            str(filtered_count),
            avg_lift,
            avg_conf
        )

    except Exception as e:
        err_df = pd.DataFrame({"Lỗi hệ thống": [str(e)]})
        return err_df, go.Figure().update_layout(title=f"Lỗi: {str(e)}"), "0", "0", "N/A", "N/A"

#---------------------------------------------------------------------
def real_admin_retrain(file_obj, retrain_cls, retrain_reg, retrain_kmeans, retrain_als):
    if file_obj is None:
        return (
            "❌ Vui lòng tải lên file dữ liệu trước!",
            go.Figure().update_layout(title="⚠️ Chưa có báo cáo"),
            pd.DataFrame()
        )

    models_selected = []
    if retrain_cls:    models_selected.append("Classification")
    if retrain_reg:    models_selected.append("Regression")
    if retrain_kmeans: models_selected.append("K-Means")
    if retrain_als:    models_selected.append("ALS")

    if not models_selected:
        return (
            "⚠️ Vui lòng chọn ít nhất một mô hình!",
            go.Figure().update_layout(title="Chưa chọn mô hình"),
            pd.DataFrame()
        )

    log_lines = []
    eval_records = []
    ts = time.strftime("%Y-%m-%d %H:%M:%S")

    def log(msg):
        log_lines.append(f"[{time.strftime('%H:%M:%S')}] {msg}")

    try:
        # ── ĐỌC FILE MỚI ──────────────────────────────────────────
        log(f"Đọc file: {file_obj.name.split('/')[-1]}")
        fname = file_obj.name

        if fname.endswith(".csv"):
            df_new = spark.read.option("header","true") \
                               .option("inferSchema","true").csv(fname)
        elif fname.endswith(".parquet"):
            df_new = spark.read.parquet(fname)
        elif fname.endswith(".json"):
            df_new = spark.read.option("multiline","true").json(fname)
        else:
            return ("❌ Định dạng file không hỗ trợ! Dùng .csv, .parquet hoặc .json",
                    go.Figure(), pd.DataFrame())

        row_count = df_new.count()
        col_count = len(df_new.columns)
        log(f"✅ Đọc thành công: {row_count:,} dòng, {col_count} cột")

        # ── CLASSIFICATION ────────────────────────────────────────
        if retrain_cls and base_pipeline is not None and classifier_model is not None:
            log("Đang đánh giá Classification trên dữ liệu mới...")
            try:
                required_cols = ['review_comment_message','order_purchase_timestamp',
                                 'order_delivered_customer_date','order_estimated_delivery_date',
                                 'payment_type','customer_state','freight_value',
                                 'product_photos_qty','product_weight_g','review_score']
                missing = [c for c in required_cols if c not in df_new.columns]

                if missing:
                    log(f"⚠️ Thiếu cột: {missing} — bỏ qua Classification")
                else:
                    # Tạo label nhị phân giống lúc train
                    df_cls = df_new.select(*required_cols).na.drop()
                    df_cls = df_cls.withColumn(
                        "label",
                        F.when(F.col("review_score") >= 4, 1.0).otherwise(0.0)
                    )
                    # Dùng pipeline đã load để transform (không train lại từ đầu)
                    transformed = base_pipeline.transform(df_cls)
                    predictions = classifier_model.transform(transformed)

                    from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
                    evaluator_acc = MulticlassClassificationEvaluator(
                        labelCol="label", predictionCol="prediction", metricName="accuracy")
                    evaluator_f1 = MulticlassClassificationEvaluator(
                        labelCol="label", predictionCol="prediction", metricName="f1")
                    evaluator_auc = BinaryClassificationEvaluator(
                        labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

                    acc  = evaluator_acc.evaluate(predictions)
                    f1   = evaluator_f1.evaluate(predictions)
                    auc  = evaluator_auc.evaluate(predictions)

                    log(f"✅ Classification — Accuracy: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
                    for metric, val, old in [("Accuracy", acc, 0.820),
                                             ("F1-Score", f1, 0.791),
                                             ("AUC-ROC",  auc, 0.847)]:
                        delta = val - old
                        eval_records.append({
                            "Mô hình": "Classification",
                            "Chỉ số": metric,
                            "Giá trị cũ": float(f"{old:.3f}"),
                            "Giá trị mới": float(f"{val:.4f}"),
                            "Thay đổi": f"{'+' if delta>=0 else ''}{delta*100:.1f}%",
                            "Trạng thái": "✅ Cải thiện" if delta >= 0 else "⚠️ Giảm"
                        })
            except Exception as e:
                log(f"⚠️ Lỗi Classification: {str(e)[:80]}")

        # ── REGRESSION ────────────────────────────────────────────
        if retrain_reg and freight_pipeline is not None:
            log("Đang đánh giá Regression trên dữ liệu mới...")
            try:
                required_cols = ['product_weight_g','distance_km','price',
                                 'product_length_cm','product_height_cm','product_width_cm',
                                 'customer_state','seller_state','freight_value']
                missing = [c for c in required_cols if c not in df_new.columns]

                if missing:
                    log(f"⚠️ Thiếu cột: {missing} — bỏ qua Regression")
                else:
                    df_reg = df_new.select(*required_cols).na.drop()
                    # Cast kiểu đúng
                    for c in ['product_weight_g','distance_km','price',
                              'product_length_cm','product_height_cm',
                              'product_width_cm','freight_value']:
                        df_reg = df_reg.withColumn(c, F.col(c).cast('double'))

                    predictions = freight_pipeline.transform(df_reg)

                    from pyspark.ml.evaluation import RegressionEvaluator
                    eval_rmse = RegressionEvaluator(
                        labelCol="freight_value", predictionCol="prediction", metricName="rmse")
                    eval_mae  = RegressionEvaluator(
                        labelCol="freight_value", predictionCol="prediction", metricName="mae")
                    eval_r2   = RegressionEvaluator(
                        labelCol="freight_value", predictionCol="prediction", metricName="r2")

                    rmse = eval_rmse.evaluate(predictions)
                    mae  = eval_mae.evaluate(predictions)
                    r2   = eval_r2.evaluate(predictions)

                    log(f"✅ Regression — RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")
                    eval_records.append({"Mô hình":"Regression","Chỉ số":"RMSE",
                        "Giá trị cũ":5.150,"Giá trị mới":float(f"{rmse:.4f}"),
                        "Thay đổi":f"{(5.150-rmse)/5.150*100:.1f}% ↓",
                        "Trạng thái":"✅ Cải thiện" if rmse < 5.150 else "⚠️ Giảm"})
                    eval_records.append({"Mô hình":"Regression","Chỉ số":"R²",
                        "Giá trị cũ":0.731,"Giá trị mới":float(f"{r2:.4f}"),
                        "Thay đổi":f"{'+' if r2>=0.731 else ''}{(r2-0.731)*100:.1f}%",
                        "Trạng thái":"✅ Cải thiện" if r2 >= 0.731 else "⚠️ Giảm"})
            except Exception as e:
                log(f"⚠️ Lỗi Regression: {str(e)[:80]}")

        # ── K-MEANS ───────────────────────────────────────────────
        if retrain_kmeans and kmeans_pipeline is not None:
            log("Đang đánh giá K-Means trên dữ liệu mới...")
            try:
                rev_col = 'payment_value' if 'payment_value' in df_new.columns else \
                          ('price' if 'price' in df_new.columns else None)

                if rev_col is None or 'order_purchase_timestamp' not in df_new.columns:
                    log("⚠️ Thiếu cột RFM — bỏ qua K-Means")
                else:
                    max_date = df_new.select(F.max('order_purchase_timestamp')).collect()[0][0]
                    rfm = df_new.na.drop(subset=['customer_unique_id','order_purchase_timestamp',rev_col]) \
                        .groupBy('customer_unique_id').agg(
                            F.datediff(F.lit(max_date), F.max('order_purchase_timestamp')).cast('int').alias('Recency'),
                            F.countDistinct('order_id').cast('int').alias('Frequency'),
                            F.sum(rev_col).cast('double').alias('Monetary')
                        ).na.drop()

                    rfm = rfm.withColumn('Log_R', F.log(F.col('Recency')+1)) \
                             .withColumn('Log_F', F.log(F.col('Frequency')+1)) \
                             .withColumn('Log_M', F.log(F.col('Monetary')+1))

                    # ---- ĐOẠN MỚI THÊM VÀO ĐỂ GOM CỘT ĐÂY NÈ ----
                    from pyspark.ml.feature import VectorAssembler
                    assembler = VectorAssembler(inputCols=['Log_R', 'Log_F', 'Log_M'], outputCol='scaled_features')
                    rfm = assembler.transform(rfm)
                    # ---------------------------------------------

                    preds = kmeans_pipeline.transform(rfm)

                    from pyspark.ml.evaluation import ClusteringEvaluator
                    evaluator = ClusteringEvaluator(
                        featuresCol='scaled_features',
                        predictionCol='Cluster_KMeans',
                        metricName='silhouette'
                    )
                    silhouette = evaluator.evaluate(preds)
                    log(f"✅ K-Means — Silhouette: {silhouette:.4f}")

                    eval_records.append({
                        "Mô hình": "K-Means", "Chỉ số": "Silhouette",
                        "Giá trị cũ": 0.550, "Giá trị mới": float(f"{silhouette:.4f}"),
                        "Thay đổi": f"{'+' if silhouette>=0.55 else ''}{(silhouette-0.55)*100:.1f}%",
                        "Trạng thái": "✅ Cải thiện" if silhouette >= 0.55 else "⚠️ Giảm"
                    })
            except Exception as e:
                log(f"⚠️ Lỗi K-Means: {str(e)[:80]}")

        # ── KẾT QUẢ ───────────────────────────────────────────────
        if not eval_records:
            log("⚠️ Không đánh giá được mô hình nào (kiểm tra lại tên cột trong file)")
            return ("\n".join(log_lines), go.Figure().update_layout(title="Không có kết quả"), pd.DataFrame())

        eval_df = pd.DataFrame(eval_records)
        log(f"🏁 Hoàn tất lúc {time.strftime('%H:%M:%S')} — {len(eval_records)} chỉ số đã đánh giá")

        # ── VẼ BIỂU ĐỒ ────────────────────────────────────────────
        chart_map = {
            'Classification': ('Accuracy','Giá trị cũ','Giá trị mới'),
            'Regression':     ('R²','Giá trị cũ','Giá trị mới'),
            'K-Means':        ('Silhouette','Giá trị cũ','Giá trị mới'),
        }
        chart_data = []
        for model, (metric, old_col, new_col) in chart_map.items():
            row = eval_df[(eval_df['Mô hình']==model) & (eval_df['Chỉ số']==metric)]
            if not row.empty:
                chart_data.append((model, row[old_col].values[0], row[new_col].values[0]))

        fig = go.Figure()
        if chart_data:
            fig.add_trace(go.Bar(
                name='Cũ', x=[d[0] for d in chart_data], y=[d[1] for d in chart_data],
                marker_color='#b5d4f4', text=[f"{d[1]:.3f}" for d in chart_data],
                textposition='outside'))
            fig.add_trace(go.Bar(
                name='Mới (Dữ liệu upload)', x=[d[0] for d in chart_data], y=[d[2] for d in chart_data],
                marker_color='#1d6bdd', text=[f"{d[2]:.4f}" for d in chart_data],
                textposition='outside'))
            fig.update_layout(
                title="📊 Đánh giá thật trên dữ liệu mới upload",
                barmode='group', template="plotly_white",
                yaxis=dict(range=[0,1.15], title="Điểm số"),
                legend=dict(orientation="h", y=1.1),
                margin=dict(l=20,r=20,t=60,b=30)
            )

        status = f"✅ ĐÁNH GIÁ HOÀN TẤT — {ts}\n"
        status += f"📁 File: {file_obj.name.split('/')[-1]} ({row_count:,} dòng)\n"
        status += f"🤖 Đã đánh giá: {', '.join(models_selected)}\n\n"
        status += "\n".join(log_lines)

        return status, fig, eval_df

    except Exception as e:
        log_lines.append(f"❌ LỖI NGHIÊM TRỌNG: {str(e)}")
        return ("\n".join(log_lines), go.Figure().update_layout(title="Lỗi"), pd.DataFrame())
# =====================================================================
# PHẦN 3: CÁC HÀM XỬ LÝ KHÁC
# =====================================================================

# --- 3.1 HÀM KHUYẾN NGHỊ SẢN PHẨM (ĐÃ CHUYỂN SANG PYSPARK) ---
def real_als_recommendations(input_id):
    if als_pipeline is None:
        return pd.DataFrame({"Lỗi": ["Hệ thống chưa tải được mô hình ALS thật!"]})

    if not input_id or input_id.strip() == "":
        return pd.DataFrame({"Thông báo": ["⚠️ Vui lòng nhập Customer Unique ID!"]})

    input_id = input_id.strip()

    try:
        # Lấy các mô hình từ Pipeline
        user_indexer = als_pipeline.stages[0]
        item_indexer = als_pipeline.stages[1]
        als_model = als_pipeline.stages[2]

        # Lấy danh sách ID sản phẩm thật từ item_indexer
        item_labels = item_indexer.labels

        # Biến ID khách hàng thành Spark DataFrame
        user_df = spark.createDataFrame([(input_id,)], ["customer_unique_id"])

        try:
            # Ép ID khách sang dạng số (Index)
            user_indexed = user_indexer.transform(user_df)
        except:
            return pd.DataFrame({"Thông báo": ["Khách hàng mới (chưa có lịch sử mua hàng trong hệ thống)."]})

        # Báo ALS dự đoán Top 10
        recs = als_model.recommendForUserSubset(user_indexed, 10)

        # PySpark check rỗng dùng isEmpty() thay vì .empty
        if recs.isEmpty():
            return pd.DataFrame({"Thông báo": ["Không tìm thấy gợi ý phù hợp cho khách hàng này."]})

        # Tách danh sách kết quả ra
        recs_exploded = recs.select(F.explode("recommendations").alias("rec"))

        # Lấy Index và Điểm
        recs_spark = recs_exploded.select(
            F.col("rec.item_index").cast("int").alias("idx"),
            F.round(F.col("rec.rating"), 4).alias("Điểm Gợi Ý (ALS Score)")
        )

        # DỊCH NGƯỢC BẰNG PYSPARK: Chuyển Index thành Product ID gốc
        converter = IndexToString(inputCol="idx", outputCol="product_id", labels=item_labels)
        recs_spark = converter.transform(recs_spark)

        # GỘP VỚI TỪ ĐIỂN BẰNG PYSPARK ĐỂ LẤY TÊN DANH MỤC
        if df_products_dict is not None and not df_products_dict.isEmpty():
            # Join trên môi trường Spark
            joined_df = recs_spark.join(df_products_dict, on='product_id', how='left')

            # Kéo 10 dòng kết quả về Pandas để xuất lên Gradio
            res_pd = joined_df.toPandas()

            # Sắp xếp lại theo điểm từ cao xuống thấp và tạo cột Top
            res_pd = res_pd.sort_values(by="Điểm Gợi Ý (ALS Score)", ascending=False).reset_index(drop=True)
            res_pd['Top'] = range(1, len(res_pd) + 1)

            res_pd.rename(columns={
                'product_id': 'Mã Sản Phẩm',
                'product_category_name': 'Danh Mục (Category)'
            }, inplace=True)
            return res_pd[['Top', 'Danh Mục (Category)', 'Mã Sản Phẩm', 'Điểm Gợi Ý (ALS Score)']]

        else:
            # Nếu không có từ điển sản phẩm
            res_pd = recs_spark.toPandas()
            res_pd = res_pd.sort_values(by="Điểm Gợi Ý (ALS Score)", ascending=False).reset_index(drop=True)
            res_pd['Top'] = range(1, len(res_pd) + 1)
            res_pd.rename(columns={'product_id': 'Mã Sản Phẩm'}, inplace=True)
            return res_pd[['Top', 'Mã Sản Phẩm', 'Điểm Gợi Ý (ALS Score)']]

    except Exception as e:
        return pd.DataFrame({"Lỗi hệ thống": [str(e)]})


# --- 3.2 HÀM XỬ LÝ ADMIN RETRAIN ---


# =====================================================================
# PHẦN 4: KHUNG GIAO DIỆN WEB (UI) ĐÃ CẢI TIẾN CSS
# =====================================================================

# 🌟 1. KHAI BÁO CSS TÙY CHỈNH LÀM ĐẸP GIAO DIỆN
custom_css = """
/* ============================================================
   IMPORT FONT ĐẸP HƠN
   ============================================================ */
@import url('https://fonts.googleapis.com/css2?family=Be+Vietnam+Pro:wght@400;500;600;700&display=swap');

/* ============================================================
   NỀN TỔNG THỂ
   ============================================================ */
.gradio-container {
    background: linear-gradient(135deg, #f0f4ff 0%, #fafafa 50%, #f0faf5 100%) !important;
    font-family: 'Be Vietnam Pro', sans-serif !important;
    min-height: 100vh;
}

/* ============================================================
   THANH TAB – dùng đúng selector của Gradio mới
   ============================================================ */
/* Nền thanh tab */
.tab-nav {
    background: white !important;
    border-radius: 12px !important;
    padding: 6px !important;
    box-shadow: 0 2px 12px rgba(0,0,0,0.06) !important;
    border: 1px solid #e2e8f0 !important;
    gap: 4px !important;
}

/* Từng nút tab */
.tab-nav button {
    border-radius: 8px !important;
    font-weight: 500 !important;
    font-size: 14px !important;
    padding: 8px 16px !important;
    transition: all 0.25s ease !important;
    border: none !important;
    color: #64748b !important;
    background: transparent !important;
}

/* Tab đang được chọn (active) */
.tab-nav button.selected {
    background: linear-gradient(135deg, #2563eb, #0ea5e9) !important;
    color: white !important;
    font-weight: 700 !important;
    box-shadow: 0 4px 12px rgba(37, 99, 235, 0.35) !important;
    transform: translateY(-1px) !important;
}

/* Hover tab chưa chọn */
.tab-nav button:not(.selected):hover {
    background: #f1f5f9 !important;
    color: #1e40af !important;
}

/* ============================================================
   NÚT BẤM PRIMARY (Nút xanh chính)
   ============================================================ */
button.primary, .btn.primary,
button[class*="primary"] {
    background: linear-gradient(135deg, #2563eb 0%, #0ea5e9 100%) !important;
    color: white !important;
    border: none !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
    font-size: 14px !important;
    padding: 10px 22px !important;
    box-shadow: 0 4px 14px rgba(37, 99, 235, 0.3) !important;
    transition: all 0.3s cubic-bezier(0.34, 1.56, 0.64, 1) !important;
    letter-spacing: 0.3px !important;
}
button.primary:hover, button[class*="primary"]:hover {
    transform: translateY(-3px) scale(1.02) !important;
    box-shadow: 0 8px 20px rgba(37, 99, 235, 0.4) !important;
    background: linear-gradient(135deg, #1d4ed8 0%, #0284c7 100%) !important;
}
button.primary:active, button[class*="primary"]:active {
    transform: translateY(0px) scale(0.98) !important;
}

/* ============================================================
   NÚT STOP (Nút đỏ - Retrain)
   ============================================================ */
button.stop, button[class*="stop"] {
    background: linear-gradient(135deg, #ef4444 0%, #f97316 100%) !important;
    color: white !important;
    border: none !important;
    border-radius: 10px !important;
    font-weight: 700 !important;
    font-size: 14px !important;
    padding: 10px 22px !important;
    box-shadow: 0 4px 14px rgba(239, 68, 68, 0.3) !important;
    transition: all 0.3s cubic-bezier(0.34, 1.56, 0.64, 1) !important;
    letter-spacing: 0.3px !important;
}
button.stop:hover, button[class*="stop"]:hover {
    transform: translateY(-3px) scale(1.02) !important;
    box-shadow: 0 8px 24px rgba(239, 68, 68, 0.45) !important;
    background: linear-gradient(135deg, #dc2626 0%, #ea580c 100%) !important;
}
button.stop:active, button[class*="stop"]:active {
    transform: translateY(0px) scale(0.98) !important;
}

/* ============================================================
   NÚT SECONDARY / OUTLINE (nút thường)
   ============================================================ */
button.secondary, button[class*="secondary"] {
    background: white !important;
    color: #374151 !important;
    border: 1.5px solid #d1d5db !important;
    border-radius: 10px !important;
    font-weight: 500 !important;
    transition: all 0.2s ease !important;
}
button.secondary:hover, button[class*="secondary"]:hover {
    border-color: #2563eb !important;
    color: #2563eb !important;
    background: #eff6ff !important;
    transform: translateY(-1px) !important;
}

/* ============================================================
   KPI LABEL CARDS
   ============================================================ */
.label-wrap, .output-class {
    background: white !important;
    border-radius: 14px !important;
    border: 1px solid #e2e8f0 !important;
    box-shadow: 0 2px 12px rgba(0,0,0,0.05) !important;
    transition: transform 0.2s ease, box-shadow 0.2s ease !important;
    overflow: hidden !important;
}
.label-wrap:hover, .output-class:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 6px 20px rgba(37, 99, 235, 0.1) !important;
}

/* ============================================================
   SLIDER
   ============================================================ */
input[type="range"] {
    accent-color: #2563eb !important;
    height: 6px !important;
}

/* ============================================================
   INPUT TEXT / TEXTAREA / NUMBER
   ============================================================ */
input[type="text"], input[type="number"], textarea {
    border: 1.5px solid #e2e8f0 !important;
    border-radius: 10px !important;
    font-family: 'Be Vietnam Pro', sans-serif !important;
    transition: border-color 0.2s ease, box-shadow 0.2s ease !important;
    background: white !important;
}
input[type="text"]:focus, input[type="number"]:focus, textarea:focus {
    border-color: #2563eb !important;
    box-shadow: 0 0 0 3px rgba(37, 99, 235, 0.12) !important;
    outline: none !important;
}

/* ============================================================
   DROPDOWN SELECT
   ============================================================ */
select, .wrap-inner {
    border-radius: 10px !important;
    border: 1.5px solid #e2e8f0 !important;
    font-family: 'Be Vietnam Pro', sans-serif !important;
}

/* ============================================================
   ACCORDION
   ============================================================ */
.accordion {
    border-radius: 12px !important;
    border: 1px solid #e2e8f0 !important;
    box-shadow: 0 2px 8px rgba(0,0,0,0.04) !important;
    overflow: hidden !important;
    background: white !important;
}
.accordion > .label-wrap {
    background: linear-gradient(to right, #f8fafc, #f0f4ff) !important;
    border-radius: 0 !important;
    box-shadow: none !important;
    border-bottom: 1px solid #e2e8f0 !important;
    font-weight: 600 !important;
    padding: 12px 16px !important;
}

/* ============================================================
   DATAFRAME (BẢNG)
   ============================================================ */
.dataframe-container, table {
    border-radius: 12px !important;
    overflow: hidden !important;
    border: 1px solid #e2e8f0 !important;
    box-shadow: 0 2px 8px rgba(0,0,0,0.04) !important;
}
table thead tr th {
    background: linear-gradient(to right, #1e3a8a, #1d4ed8) !important;
    color: white !important;
    font-weight: 600 !important;
    font-size: 13px !important;
    padding: 12px 14px !important;
    letter-spacing: 0.3px !important;
}
table tbody tr:nth-child(even) {
    background: #f8faff !important;
}
table tbody tr:hover {
    background: #eff6ff !important;
    transition: background 0.15s ease !important;
}
table tbody tr td {
    padding: 10px 14px !important;
    font-size: 13px !important;
    color: #374151 !important;
    border-bottom: 1px solid #f1f5f9 !important;
}

/* ============================================================
   FILE UPLOAD
   ============================================================ */
.file-preview, .upload-button {
    border-radius: 12px !important;
    border: 2px dashed #cbd5e1 !important;
    transition: all 0.2s ease !important;
    background: #f8faff !important;
}
.file-preview:hover {
    border-color: #2563eb !important;
    background: #eff6ff !important;
}

/* ============================================================
   CHECKBOX
   ============================================================ */
input[type="checkbox"] {
    accent-color: #2563eb !important;
    width: 16px !important;
    height: 16px !important;
    cursor: pointer !important;
}

/* ============================================================
   BOX CHỨA (gr.Row, gr.Column, gr-box)
   ============================================================ */
.gap, .gr-group, .form {
    background: white !important;
    border-radius: 14px !important;
    box-shadow: 0 2px 10px rgba(0,0,0,0.04) !important;
    border: 1px solid #f1f5f9 !important;
}

/* ============================================================
   PLOT (biểu đồ)
   ============================================================ */
.plot-container, .plot {
    border-radius: 14px !important;
    overflow: hidden !important;
    border: 1px solid #e2e8f0 !important;
    box-shadow: 0 2px 10px rgba(0,0,0,0.04) !important;
    background: white !important;
}

/* ============================================================
   SCROLLBAR ĐẸP HƠN
   ============================================================ */
::-webkit-scrollbar {
    width: 6px;
    height: 6px;
}
::-webkit-scrollbar-track {
    background: #f1f5f9;
    border-radius: 10px;
}
::-webkit-scrollbar-thumb {
    background: #cbd5e1;
    border-radius: 10px;
}
::-webkit-scrollbar-thumb:hover {
    background: #2563eb;
}

/* ============================================================
   ANIMATION CHO CÁC PHẦN TỬ KHI TẢI
   ============================================================ */
@keyframes fadeInUp {
    from { opacity: 0; transform: translateY(12px); }
    to   { opacity: 1; transform: translateY(0); }
}
.gradio-container > * {
    animation: fadeInUp 0.4s ease forwards;
}

/* ============================================================
   PROGRESS BAR (khi đang xử lý)
   ============================================================ */
.progress-bar {
    background: linear-gradient(90deg, #2563eb, #0ea5e9, #06b6d4) !important;
    background-size: 200% 100% !important;
    animation: shimmer 1.5s infinite !important;
    border-radius: 99px !important;
    height: 4px !important;
}
@keyframes shimmer {
    0%   { background-position: 200% center; }
    100% { background-position: -200% center; }
}
"""

# Truyền CSS vào khối Blocks
# Sửa ở phần đầu lúc tạo demo
with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:
    # ... (các component của bạn bên trong giữ nguyên) ...

    # Header Banner
    gr.HTML("""
    <div style="background: linear-gradient(135deg, #ffffff, #e0eafc); padding: 25px; border-radius: 15px; text-align: center; box-shadow: 0 4px 15px rgba(0,0,0,0.05); margin-bottom: 20px; border: 1px solid #e2e8f0;">
        <h1 style="color: #1e3a8a; font-size: 32px; margin: 0; font-weight: 900; letter-spacing: 1px;">🛒 HỆ THỐNG PHÂN TÍCH & KHUYẾN NGHỊ THƯƠNG MẠI ĐIỆN TỬ OLIST</h1>
        <p style="color: #64748b; font-size: 16px; margin-top: 10px; font-weight: 500;">Powered by PySpark Big Data Analytics & Machine Learning</p>
    </div>
    """)

    with gr.Tabs():

        # --- ------------------------------------------------ ---
        # --- TAB DASHBOARD ---
        with gr.Tab("📊 Dashboard & Thống kê"):
            gr.Markdown("## 📈 TỔNG QUAN HỆ THỐNG THƯƠNG MẠI ĐIỆN TỬ OLIST")

            # Xử lý list danh mục
            if df_master is not None and not df_master.isEmpty():
                cat_col = 'product_category_name_english' if 'product_category_name_english' in df_master.columns else ('product_category_name' if 'product_category_name' in df_master.columns else None)
                if cat_col:
                    unique_rows = df_master.select(cat_col).na.drop().distinct().collect()
                    unique_cats = [str(row[0]) for row in unique_rows]
                    real_categories = ["Tất cả"] + sorted(unique_cats)
                else:
                    real_categories = ["Tất cả"]
            else:
                real_categories = ["Tất cả"]

            with gr.Accordion("🔍 Bộ lọc Dashboard (Mở rộng để lọc)", open=True):
                with gr.Row():
                    filter_year = gr.Dropdown(choices=["Tất cả", "2016", "2017", "2018"], value="Tất cả", label="Lọc theo Năm")
                    filter_cat = gr.Dropdown(choices=real_categories, value="Tất cả", label="Lọc theo Danh mục")
                    filter_btn = gr.Button("🔄 Áp dụng Bộ lọc", variant="primary")

            with gr.Row():
                kpi_cust = gr.Label(label="👥 Tổng Khách Hàng")
                kpi_ord = gr.Label(label="📦 Tổng Đơn Hàng")
                kpi_rev = gr.Label(label="💰 Tổng Doanh Thu")

            gr.Markdown("---")

            with gr.Row():
                plot_revenue = gr.Plot()
                plot_review = gr.Plot()
            with gr.Row():
                plot_cats = gr.Plot()
                plot_payment = gr.Plot()

            gr.Markdown("---")
            gr.Markdown("### 🌌 Phân bố Cụm Khách Hàng Toàn Hệ Thống (RFM & K-Means)")
            plot_dashboard_cluster = gr.Plot()

            dashboard_outputs = [kpi_cust, kpi_ord, kpi_rev, plot_revenue, plot_review, plot_cats, plot_payment, plot_dashboard_cluster]

            filter_btn.click(fn=load_filtered_dashboard, inputs=[filter_year, filter_cat], outputs=dashboard_outputs)

        # --- ------------------------------------------------ ---
        # --- TAB CLUSTERING ---
        with gr.Tab("👥 Phân khúc (Clustering)"):
            gr.Markdown("### Tải lên dữ liệu khách hàng (CSV) để phân nhóm")
            with gr.Row():
                cluster_input = gr.File(label="Upload File Khách hàng chứa cột RFM (CSV)")
                cluster_btn = gr.Button("👥 Bắt đầu Phân cụm", variant="primary")

            gr.Markdown("### 📊 Đặc điểm Hành vi của từng Nhóm Khách hàng")
            cluster_summary_table = gr.Dataframe(label="Phân tích Đặc điểm Nhóm")

            gr.Markdown("---")
            with gr.Row():
                cluster_plot = gr.Plot(label="Biểu đồ phân bố khách hàng 3D")
                cluster_output_table = gr.Dataframe(label="Kết quả Phân cụm chi tiết (Top 5000)")

            cluster_btn.click(fn=real_clustering, inputs=cluster_input, outputs=[cluster_output_table, cluster_plot, cluster_summary_table])

        # --- ------------------------------------------------ ---
        # --- TAB KHUYẾN NGHỊ ALS ---
        with gr.Tab("🎁 Khuyến nghị (ALS)"):
            gr.Markdown("### Gợi ý sản phẩm cho Khách hàng dựa trên lịch sử mua sắm")
            with gr.Row():
                als_id_input = gr.Textbox(label="Nhập Customer Unique ID vào đây", placeholder="VD: 3e43e6105506432c953e165fb2acf44c")
                recommend_btn = gr.Button("🔍 Trích xuất Top 10 Sản Phẩm", variant="primary")

            recommend_output = gr.Dataframe(label="Top 10 Sản phẩm đề xuất từ mô hình ALS")
            recommend_btn.click(fn=real_als_recommendations, inputs=als_id_input, outputs=recommend_output)

        # --- ------------------------------------------------ ---
        # --- TAB FP-GROWTH ---
        with gr.Tab("🛒 Xu hướng (FP-Growth)"):
            gr.Markdown("### 🛒 Khám phá sản phẩm thường được mua cùng nhau (FP-Growth)")

            # KPI Row
            with gr.Row():
                kpi_total_rules = gr.Label(label="📋 Tổng số luật")
                kpi_filtered_rules = gr.Label(label="🔍 Luật sau lọc")
                kpi_avg_lift = gr.Label(label="📈 Lift trung bình")
                kpi_avg_conf = gr.Label(label="🎯 Confidence trung bình")

            with gr.Accordion("🔧 Bộ lọc nâng cao", open=True):
                with gr.Row():
                    conf_slider = gr.Slider(
                        minimum=0.0, maximum=1.0, value=0.05, step=0.01,
                        label="Confidence tối thiểu"
                    )
                    lift_slider = gr.Slider(
                        minimum=1.0, maximum=10.0, value=1.2, step=0.1,
                        label="Lift tối thiểu"
                    )
                    fp_search = gr.Textbox(
                        label="🔎 Tìm kiếm sản phẩm (antecedent/consequent)",
                        placeholder="Nhập tên danh mục...",
                        value=""
                    )

            fp_output = gr.Dataframe(
                label="📊 Luật kết hợp (Top 50, sắp xếp theo Lift giảm dần)",
                wrap=True
            )
            fp_lift_plot = gr.Plot(label="Phân phối Lift của các luật sau lọc")

            # Event: cập nhật khi thay đổi bất kỳ bộ lọc nào
            fp_inputs = [conf_slider, lift_slider, fp_search]
            fp_outputs = [fp_output, fp_lift_plot, kpi_total_rules, kpi_filtered_rules, kpi_avg_lift, kpi_avg_conf]

            conf_slider.change(fn=real_fp_rules, inputs=fp_inputs, outputs=fp_outputs)
            lift_slider.change(fn=real_fp_rules, inputs=fp_inputs, outputs=fp_outputs)
            fp_search.change(fn=real_fp_rules, inputs=fp_inputs, outputs=fp_outputs)

        # --- ------------------------------------------------ ---
        # --- TAB DỰ ĐOÁN ML ---
        with gr.Tab("🔮 Dự đoán Đơn hàng"):
            gr.Markdown("### 1. Dự đoán Điểm Đánh Giá (Review Score)")
            with gr.Row():
                with gr.Column():
                    class_comment = gr.Textbox(lines=2, label="Nội dung bình luận (nếu có)")
                    class_payment = gr.Dropdown(["credit_card", "boleto", "voucher", "debit_card"], label="Phương thức thanh toán", value="credit_card")
                    class_state = gr.Textbox(label="Mã Bang Khách hàng (VD: SP)", value="SP")
                    class_freight = gr.Number(label="Phí vận chuyển (BRL)", value=15.0)
                    with gr.Row():
                        class_photos = gr.Number(label="Số ảnh", value=1.0)
                        class_weight = gr.Number(label="Trọng lượng (g)", value=500.0)

                with gr.Column():
                    class_purchase = gr.Textbox(label="Thời gian đặt hàng", value="2018-05-01 10:00:00")
                    class_deliver = gr.Textbox(label="Thời gian giao thực tế", value="2018-05-05 14:00:00")
                    class_estimate = gr.Textbox(label="Thời gian dự kiến", value="2018-05-10 23:59:00")
                    class_btn = gr.Button("🔮 Dự đoán Điểm Đánh Giá", variant="primary")
                    class_output = gr.Label(label="Kết quả dự đoán (Review Score)")

            class_btn.click(fn=real_classify_review, inputs=[class_comment, class_purchase, class_deliver, class_estimate, class_payment, class_state, class_freight, class_photos, class_weight], outputs=class_output)

            gr.Markdown("---")
            gr.Markdown("### 2. Dự đoán Phí Vận Chuyển (Freight Value)")
            with gr.Row():
                product_weight = gr.Number(label="Trọng lượng (g)", value=500)
                distance = gr.Number(label="Khoảng cách (km)", value=100)
                product_price = gr.Number(label="Giá sản phẩm (BRL)", value=100)

            with gr.Accordion("📦 Thông số đóng gói", open=False):
                with gr.Row():
                    p_length = gr.Number(label="Dài (cm)", value=30)
                    p_height = gr.Number(label="Cao (cm)", value=30)
                    p_width = gr.Number(label="Rộng (cm)", value=30)
                    cust_state = gr.Textbox(label="Mã Bang Khách", value="SP")
                    sell_state = gr.Textbox(label="Mã Bang Shop", value="SP")

            with gr.Row():
                reg_btn = gr.Button("🚚 Dự đoán Freight Value", variant="primary")
                reg_output = gr.Textbox(label="Kết quả Phí Ship dự kiến (BRL)")

            reg_btn.click(fn=real_predict_freight, inputs=[product_weight, distance, product_price, p_length, p_height, p_width, cust_state, sell_state], outputs=reg_output)

        # --- ------------------------------------------------ ---
        # --- TAB ADMIN ---
        with gr.Tab("⚙️ Quản trị (Admin)"):
            gr.Markdown("## 🛠️ Cập nhật Dữ liệu & Huấn luyện lại Mô hình")
            gr.Markdown("> Tải lên Master Data mới, chọn mô hình cần retrain, xem báo cáo đánh giá và so sánh hiệu suất.")

            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 📁 Bước 1: Tải dữ liệu mới")
                    admin_file_input = gr.File(
                        label="Chọn file Master Data mới (.parquet hoặc .csv)",
                        file_types=[".parquet", ".csv", ".json"]
                    )

                    gr.Markdown("### 🤖 Bước 2: Chọn mô hình cần retrain")
                    retrain_cls = gr.Checkbox(label="Phân loại điểm đánh giá (Classification)", value=True)
                    retrain_reg = gr.Checkbox(label="Dự đoán phí vận chuyển (GBT Regression)", value=True)
                    retrain_kmeans = gr.Checkbox(label="Phân cụm khách hàng (K-Means)", value=True)
                    retrain_als = gr.Checkbox(label="Khuyến nghị sản phẩm (ALS)", value=False)

                    admin_retrain_btn = gr.Button(
                        "🚀 Khởi chạy Retrain Mô hình",
                        variant="stop"
                    )

                with gr.Column(scale=2):
                    gr.Markdown("### 📊 Bước 3: Xem kết quả & Báo cáo")
                    admin_status_output = gr.Textbox(
                        label="Nhật ký huấn luyện",
                        lines=12,
                        placeholder="Kết quả retrain sẽ hiện ở đây sau khi hoàn tất..."
                    )

            gr.Markdown("---")
            gr.Markdown("### 📈 Biểu đồ So sánh Hiệu suất (Cũ vs Mới)")
            admin_eval_plot = gr.Plot(label="Model Performance Report")

            gr.Markdown("### 📋 Bảng Đánh giá Chi tiết theo từng Chỉ số")
            admin_eval_table = gr.Dataframe(
                label="Kết quả đánh giá đầy đủ",
                headers=["Mô hình", "Chỉ số", "Giá trị cũ", "Giá trị mới", "Thay đổi", "Trạng thái"]
            )

            admin_retrain_btn.click(
                fn=real_admin_retrain,
                inputs=[admin_file_input, retrain_cls, retrain_reg, retrain_kmeans, retrain_als],
                outputs=[admin_status_output, admin_eval_plot, admin_eval_table]
            )

    # Footer
    gr.HTML("""
    <div style="text-align: center; margin-top: 40px; padding: 20px; border-top: 1px solid #e2e8f0; color: #64748b; font-size: 14px;">
        <p>Đồ án Hệ thống Dữ liệu lớn (Big Data) • Phát triển trên nền tảng <b>PySpark</b>, <b>Gradio</b> và <b>Plotly</b>.</p>
        <p>© 2024 - Giao diện tùy chỉnh chuẩn E-commerce.</p>
    </div>
    """)

    # --- SỰ KIỆN TỰ ĐỘNG CHẠY KHI MỞ APP ---
    demo.load(fn=load_filtered_dashboard, inputs=[filter_year, filter_cat], outputs=dashboard_outputs)

    # Đã gộp và cập nhật hàm load theo yêu cầu của bạn
    demo.load(
        fn=real_fp_rules,
        inputs=[conf_slider, lift_slider, fp_search],
        outputs=[fp_output, fp_lift_plot, kpi_total_rules, kpi_filtered_rules, kpi_avg_lift, kpi_avg_conf]
    )

# =====================================================================
# PHẦN 5: CHẠY ỨNG DỤNG
# =====================================================================
# Đoạn code cũ bị lỗi:
# demo.launch(debug=True, share=True, theme=gr.themes.Soft(), css=custom_css)

# Đoạn code mới đã sửa:
if __name__ == "__main__":
    print("🚀 Đang khởi động WebUI với giao diện mới. Vui lòng chờ link Public...")
    demo.launch(debug=True, share=True)

⏳ Đang khởi tạo Spark Session...
✅ Spark đã khởi tạo xong!
⏳ Đang tải Master Data cho Dashboard...
✅ Đã tải thành công Master Data! (Tổng số dòng: 116068)
⏳ Đang tải mô hình K-Means Clustering...
✅ Đã tải thành công mô hình Clustering!
⏳ Đang tải dữ liệu Luật kết hợp (FP-Growth)...
✅ Đã tải thành công dữ liệu FP-Growth!
⏳ Đang tải mô hình Dự đoán Review Score...
✅ Đã tải thành công bộ đôi mô hình Classification!
⏳ Đang tải mô hình Dự đoán Phí Vận Chuyển (Regression)...
✅ Đã tải thành công mô hình Regression!
⏳ Đang tải mô hình Khuyến nghị (ALS)...
✅ Đã tải thành công mô hình ALS!

⏳ Đang tải Từ điển Sản phẩm (để hiện tên Danh mục)...
✅ Đã tải thành công Từ điển Sản phẩm!


/tmp/ipykernel_2200/1666470537.py:1227: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:
/tmp/ipykernel_2200/1666470537.py:1227: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:


🚀 Đang khởi động WebUI với giao diện mới. Vui lòng chờ link Public...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://067f58b393813f7dc2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
